# E7 step 3b Â· Three follow-up checks on C4's outputs (PC, no GPU)

Task **E7** (scoring our system's chain as one), step 3b. Built 19 Nov 2026 at Abrar's request ("do all three asap"), after [E7 step 3](E7-results/step3/README.md) made C4's building table and hot spots for the 2,065 test photos.

**Words used here.**
- **C4** makes the damage map: R11, our pixel model (a U-Net reading the before- and after-photo), on the coarser xBD photos; and find-then-grade (SAM 3 outlines each building, then our grader labels it intact, damaged, destroyed or "not a building") on the sharper photos, such as Turkey's.
- **The building table** (`buildings_c4.sqlite`, what the reports read): on the sharper photos one row per building SAM 3 outlined; on xBD photos one row per connected patch of R11's pixels.
- **Hot spots** ("where to search first", T1 v3's code): up to 3 zones per photo where the map's damage is densest, each with a marked point. A hot spot is a **hit** when its marked point is within 20 px of a truly damaged or destroyed building.
- **Clean pairs:** the 2,007 test pairs (E6) and the 605 validation pairs (E5) with trusted true masks and no copies.

## Check A: counting from the table's rows
- **Question:** E7 step 3's counting (34.89% on E5's clean questions) used E7 step 2's saved counts, which join touching pixels into one blob. The reports will count the table's rows instead, and on the sharper photos those are one per building. How well do the rows count?
- **How:** for each of the 3,580 askable optical counting questions, the number of the table's rows of the asked class on that photo, then the nearest answer option (E7 step 2's rule). Accuracy, count error and the share within 20% of the true count, against E7 step 3's C4, R11 and find-then-grade, with exact McNemar on the same questions.
- **Before fixing this expectation,** E7 step 2's saved counts were looked at: on the sharper photos' 1,292 questions, the joined-blob counts fall below the true count 68.5% of the time (median count / truth 0.70).

## Check B: why the hot spots fall short on photos without destroyed buildings
- **Question:** E7 step 3 found 56.9% of marked points near real damage on the test pairs, against 73.6% for T1 v3 on validation with R9's maps. On photos without destroyed buildings it was 28.7% against 58.2%. Is that C4's maps (more false damage) or the test photos (less true damage to land on)?
- **How:** C4's maps are rebuilt for the 605 clean validation pairs (R11's saved validation maps on xBD photos; SAM 3's validation outlines painted with grader seed 42 on the others), and T1 v3's code runs on them. Three sets are compared, photo group by photo group (with destroyed buildings; with only damaged ones; with no true damage at all):
  - R9's maps on validation (T1 v3's saved results);
  - C4's maps on the same validation photos;
  - C4's maps on the test photos (E7 step 3's saved results).

  From R9 on validation to C4 on validation, only the maps change; from C4 on validation to C4 on test, only the photos change. On photos without destroyed buildings, the change in the hit share at each stage measures each cause.
- **Reading, fixed before the run:** the larger of the two changes is the main cause; if both are larger than 0.05, both are.

## Check C: a stricter rule for when a photo gets hot spots at all (T1 v4)
- **Question:** most hot spots on photos without destroyed buildings are false alarms. Can a simple rule drop them while keeping the real ones?
- **The rule:** a photo gets hot spots only if its map marks at least G pixels' worth of damage (destroyed pixels count 1, damaged 0.5, as T1's danger map weighs them). Below that, the photo gets no hot spots and no safe areas, as T1 v3 does for a photo with no damage.
- **Chosen on validation only,** on C4's maps of the 605 clean validation pairs: G is the largest value on the grid 0, 100, 200, 400, 800, 1,600, 3,200, 6,400 and 12,800 px that keeps at least 95% of the validation hits.
- **Confirmed once on the 2,007 clean test pairs.** **v4 passes if** it keeps at least 90% of the test hits and raises the hit share by at least 0.05 (from 0.569 to 0.619 or more). If it passes, the gated hot-spot file is written (`hotspots_v4.json`) for the reports; switching T2 step 5 to it is Abrar's call.

## Checks (the notebook stops if one fails)
1. E7 step 2's seed-42 table, counted row by row, gives the saved seed-42 counts on every askable question; C4's table gives R11's saved counts on every question on an xBD photo; the answer options and right answers agree with the benchmark's file.
2. C4's rebuilt validation maps give E7 step 2c's validation scores within 0.0005: R11 with TTA 0.2692 destroyed IoU, 0.5805 mIoU; C4 with seed 42 0.3574 destroyed IoU (605 clean pairs).
3. T1 v3's code, run here on R9's validation maps for 60 photos, gives T1 v3's saved per-photo results exactly.
4. The gate's self-checks: G = 0 changes nothing; a photo below G loses all its hot spots; the choice rule picks the right G on a toy example.

## Expectation (fixed before the run)
- All four checks pass.
- **A:** on the sharper photos' 1,292 questions, the row counts sit closer to the truth than the joined blobs (median count / truth up from 0.70 to about 0.8â€“0.9), and accuracy there rises by 1â€“4 points. Overall, C4's accuracy on E5's clean questions rises by about 0.5â€“1.5 points (to about 35.4â€“36.4%).
- **B:** mostly the maps. On the same validation photos without destroyed buildings, C4's hot spots are hits clearly less often than R9's (below 0.45), and C4 marks more hot spots per such photo; the test photos differ less.
- **C:** on validation, G lands between 400 and 3,200 px. On the test pairs the hit share rises to about 0.60â€“0.68, with at least 90% of the hits kept. Whether that clears the +0.05 bar is uncertain.

**How to run:** on the PC, from `Essential_task_delegation/Champion_Task/`, in Jupyter with the `cse499` kernel. With `SMOKE_TEST = True` it uses 40 validation photos and 100 test photos and skips check 2. The full run takes about 15â€“25 min, estimated.

**Outputs** in `E7-results/step3b/`: `README.md`, `config.json`, `counting_from_table.csv`, `counting_per_question.csv`, `hotspot_groups.csv`, `hotspot_causes.csv`, `val_c4_hotspots_per_image.csv`, `gate_validation.csv`, `gate_test.csv`, and `hotspots_v4.json` if v4 passes.

In [1]:
# â”€â”€ Cell 1: Configuration, inputs and functions (PC, no GPU, nothing downloaded) â”€â”€
import ast
import json
import os
import re
import sqlite3
import sys
import time
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.stats import binomtest
from tqdm.auto import tqdm

SMOKE_TEST = False             # True = 40 validation photos and 100 test photos; check 2 needs all 605 clean validation pairs
CT = Path(os.environ.get("CHAMPION_TASK", Path.cwd()))     # run from Champion_Task
OUT_DIR = Path(os.environ.get("E7S3B_OUT", CT / "E7-results" / "step3b"))
S2, S2C, S3 = CT / "E7-results" / "step2", CT / "E7-results" / "step2c", CT / "E7-results" / "step3"
E6 = CT / "E6-results" / "E6"
BENCH_JSON = CT.parents[1] / "support" / "Abrar" / "Implementation_DisasterM3" / "benchmark_release.json"
VAL_GT, VAL_R9 = CT / "E1-results" / "E1" / "gt", CT / "E1-results" / "E1" / "preds" / "R9_tta"
VAL_R11, VAL_SAM = CT / "R16-results" / "R16" / "R11_tta", CT / "E3-results" / "E3" / "masks" / "pre_grid2"
STEP5, LEAKS, T1V3 = CT / "R16c-results" / "step5", CT / "E5-results" / "E5" / "leaked_pairs_40.csv", CT / "T1-results" / "T1-v3"
INPUTS = [S3 / "buildings_c4.sqlite", S3 / "hotspots.json", S3 / "hotspots_per_image.csv", S3 / "counting.csv",
          S2 / "buildings_chain_s42.sqlite", S2 / "per_question.csv", S2C / "candidates_validation.csv",
          E6 / "test_pairs.csv", E6 / "test_masks_4class.zip", BENCH_JSON, VAL_GT, VAL_R9, VAL_R11, VAL_SAM,
          STEP5 / "instances_val.csv", STEP5 / "calibration.json", LEAKS, T1V3 / "T1_per_image.csv"]
missing = [str(p) for p in INPUTS if not p.exists()]
assert not missing, f"Run this notebook from Champion_Task; missing: {missing}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 4
CLASS_NAME = {1: "intact", 2: "damaged", 3: "destroyed"}
CLASS_OF = {"Intact building counting.": 1, "Damaged building counting.": 2, "Totally-destroyed building counting.": 3}
LABELS = ["intact", "damaged", "destroyed", "none"]          # the graders' four labels, in their order
TABLE_SEED, BIAS_42 = 42, [0.0, -1.75, -4.0, -1.25]          # E7 step 2's fixed biases for seed 42
TOL = 0.0005
# T1 v3's settings, unchanged
SIGMA_PX, DAMAGED_WEIGHT, SAFE_SCALE_PX, HOT_THRESHOLD, MIN_SPOT_PX = 25, 0.5, 200, 0.05, 500
POINT_EDGE_MARGIN_PX, POINT_RADIUS_PX = 32, 20
GATE_GRID = [0, 100, 200, 400, 800, 1600, 3200, 6400, 12800]   # check C's grid, px of damage (destroyed 1, damaged 0.5)
KEEP_HITS_VAL, KEEP_HITS_TEST, MIN_GAIN = 0.95, 0.90, 0.05
V3_TEST_SHARE = 0.569                                         # E7 step 3, clean test pairs
N_T1_CHECK = 60
XBD_NAME = re.compile(r"_\d{8}_post_disaster")


def source_of(name):
    """E1's rule: xBD photos carry an 8-digit tile number before '_post_disaster'."""
    return "xBD" if XBD_NAME.search(name) else "other"


def read_png(path_or_zip, name=None, flags=cv2.IMREAD_UNCHANGED):
    data = path_or_zip.read(name) if name is not None else Path(path_or_zip).read_bytes()
    return cv2.imdecode(np.frombuffer(data, np.uint8), flags)


def confusion(gt, pred):
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)


def ious(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    iou = tp / (cm.sum(0) + cm.sum(1) - tp + 1e-8)
    return {"miou": round(float(iou.mean()), 4), "destroyed_iou": round(float(iou[3]), 4)}


# T1 v3's map and hot-spot functions, unchanged (as in E7 step 3)
EIGHT = np.ones((3, 3), bool)


def survivability_maps(mask):
    weight = (mask == 3).astype(np.float32) + DAMAGED_WEIGHT * (mask == 2)
    danger = ndimage.gaussian_filter(weight, SIGMA_PX, mode="mirror", truncate=4.0)
    damage = mask >= 2
    dist = (ndimage.distance_transform_edt(~damage).astype(np.float32) if damage.any()
            else np.full(mask.shape, np.inf, np.float32))
    safety = np.minimum(dist / SAFE_SCALE_PX, 1).astype(np.float32)
    return danger, safety, danger * (1 - safety), dist


def hot_spots(priority, k=3):
    labels, n = ndimage.label(priority >= HOT_THRESHOLD, structure=EIGHT)
    if n == 0:
        return [], labels
    area = np.bincount(labels.ravel(), minlength=n + 1)
    mass = np.bincount(labels.ravel(), weights=priority.ravel(), minlength=n + 1)
    mass[0] = -1
    mass[area < MIN_SPOT_PX] = -1
    h, w = priority.shape
    m = POINT_EDGE_MARGIN_PX
    spots = []
    for i in np.argsort(-mass)[:k]:
        if mass[i] < 0:
            break
        ys, xs = np.nonzero(labels == i)
        inner = (ys >= m) & (ys < h - m) & (xs >= m) & (xs < w - m)
        if inner.any():
            ys, xs = ys[inner], xs[inner]
        values = priority[ys, xs]
        top = values >= values.max() - 1e-6
        j = np.argmin((ys[top] - ys.mean()) ** 2 + (xs[top] - xs.mean()) ** 2)
        spots.append({"label": int(i), "point_xy": [int(xs[top][j]), int(ys[top][j])]})
    return spots, labels


def near(gt, xy, classes):
    x, y, r = xy[0], xy[1], POINT_RADIUS_PX
    return bool(np.isin(gt[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1], classes).any())


def spot_hits(pred, gt):
    """T1 v3's hot spots on one map, each with its ground-truth checks."""
    spots, labels = hot_spots(survivability_maps(pred)[2])
    out = []
    for s in spots:
        zone = labels == s["label"]
        out.append({"on_gt_destroyed": bool((zone & (gt == 3)).any()), "on_gt_damage": bool((zone & (gt >= 2)).any()),
                    "point_on_gt_destroyed": near(gt, s["point_xy"], [3]), "point_on_gt_damage": near(gt, s["point_xy"], [2, 3])})
    return out


def damage_mass(pred_destroyed_px, pred_damaged_px):
    """Check C's measure: pixels' worth of damage, weighted as T1's danger map weighs them."""
    return pred_destroyed_px + DAMAGED_WEIGHT * pred_damaged_px


def gate_table(spots, grid):
    """For each G: hot spots kept (photo mass >= G), hits kept, hit share. `spots`: one row per hot spot."""
    rows = []
    for g in grid:
        kept = spots[spots["mass"] >= g]
        rows.append({"gate_px": g, "hot_spots": len(kept), "hits": int(kept["hit"].sum()),
                     "hit_share": round(float(kept["hit"].mean()), 3) if len(kept) else float("nan"),
                     "photos_with_hot_spots": kept["image"].nunique()})
    t = pd.DataFrame(rows)
    t["hits_kept"] = (t["hits"] / max(int(t.loc[t["gate_px"] == 0, "hits"].iloc[0]), 1)).round(3)
    return t


def choose_gate(table, keep):
    """The largest G that keeps at least `keep` of the hits (G = 0 keeps all)."""
    return int(table.loc[table["hits_kept"] >= keep, "gate_px"].max())


# Check 4: the gate's self-checks
_s = pd.DataFrame({"image": ["a", "a", "b", "c"], "mass": [50.0, 50.0, 500.0, 5000.0], "hit": [0, 0, 1, 1]})
_t = gate_table(_s, [0, 100, 1000, 10000])
assert _t.loc[0, "hot_spots"] == 4 and _t.loc[0, "hits_kept"] == 1.0          # G = 0 changes nothing
assert _t.loc[1, "hot_spots"] == 2 and _t.loc[1, "photos_with_hot_spots"] == 2  # photo "a" (mass 50) loses both spots
assert choose_gate(_t, 0.95) == 100 and choose_gate(_t, 0.5) == 1000
assert damage_mass(10, 4) == 12.0
print(f"âœ“ E7 step 3b{'  [SMOKE TEST]' if SMOKE_TEST else ''}: inputs found; check 4 (the gate's self-checks) passed")
print(f"  Outputs: {OUT_DIR}")
t_start = time.time()

âœ“ E7 step 3b: inputs found; check 4 (the gate's self-checks) passed
  Outputs: D:\CSE499AB_project\Essential_task_delegation\Champion_Task\E7-results\step3b


In [2]:
# â”€â”€ Cell 2: Check A, counting from the table's rows (and check 1) â”€â”€
def options_of(e):
    o = e["options_list"]
    return [float(str(x).rstrip("%")) for x in (ast.literal_eval(o) if isinstance(o, str) else o)]


def nearest_option(n, options):
    """E7 step 2's (H1 V1's) rule: the option closest to the count, the first one on a tie."""
    return "ABCDE"[min(range(len(options)), key=lambda i: abs(options[i] - n))]


with open(BENCH_JSON, encoding="utf-8") as f:
    bench = json.load(f)
bdc = [e for e in bench if e.get("task") == "Building Damage Counting"]
del bench
OPTIONS = {f"building_damage_counting_{i}": options_of(e) for i, e in enumerate(bdc)}     # E7 step 2's ids
RIGHT = {f"building_damage_counting_{i}": e["ground_truth_option"].strip()[0] for i, e in enumerate(bdc)}

q = pd.read_csv(S2 / "per_question.csv")
q = q[q["R11 | count"].notna() & q["chain_s42 | count"].notna()].copy()
assert len(q) == 3580, f"{len(q)} askable questions, not 3,580"
assert (q["id"].map(RIGHT) == q["truth_option"]).all(), "check 1 failed: per_question.csv's right answers differ from the benchmark's"
q["options"] = q["id"].map(OPTIONS)
q["source"] = q["image"].map(source_of)


def row_counts(db):
    conn = sqlite3.connect(f"file:{db}?mode=ro", uri=True)
    df = pd.read_sql("SELECT image, class_id, COUNT(*) AS n FROM buildings GROUP BY image, class_id", conn)
    conn.close()
    return {(r.image, int(r.class_id)): int(r.n) for r in df.itertuples()}


S42_ROWS, C4_ROWS = row_counts(S2 / "buildings_chain_s42.sqlite"), row_counts(S3 / "buildings_c4.sqlite")
q["chain_s42 rows | count"] = [S42_ROWS.get((im, int(c)), 0) for im, c in zip(q["image"], q["class"])]
q["C4 table | count"] = [C4_ROWS.get((im, int(c)), 0) for im, c in zip(q["image"], q["class"])]
xbd = q["source"] == "xBD"
assert (q["chain_s42 rows | count"] == q["chain_s42 | count"]).all(), "check 1 failed: seed 42's table doesn't give its saved counts"
assert (q.loc[xbd, "C4 table | count"] == q.loc[xbd, "R11 | count"]).all(), "check 1 failed: C4's xBD rows don't give R11's counts"
for m in ("R11", "chain_s42"):
    redo = [nearest_option(n, o) == t for n, o, t in zip(q[f"{m} | count"], q["options"], q["truth_option"])]
    assert (np.array(redo) == q[f"{m} | correct"].astype(bool).to_numpy()).all(), f"check 1 failed: {m}'s answers don't reproduce"
q["C4 blobs | count"] = np.where(xbd, q["R11 | count"], q["chain_s42 | count"])            # E7 step 3's C4
q["C4 blobs | correct"] = np.where(xbd, q["R11 | correct"], q["chain_s42 | correct"]).astype(bool)
q["C4 table | correct"] = [nearest_option(n, o) == t for n, o, t in zip(q["C4 table | count"], q["options"], q["truth_option"])]
print(f"âœ“ Check 1: seed 42's table gives its saved counts on all {len(q):,} questions, C4's xBD rows give R11's on all "
      f"{int(xbd.sum()):,} xBD questions, and both methods' answers reproduce")

METHODS = ["C4 table", "C4 blobs", "R11", "chain_s42"]


def stats(df, m):
    err = (df[f"{m} | count"] - df["truth"]).abs()
    ok = df[f"{m} | correct"].astype(bool)
    return {"method": m, "questions": len(df), "accuracy_%": round(100 * ok.mean(), 2),
            "largest_option_up_to_20_%": round(100 * ok[df["largest_option"] <= 20].mean(), 2),
            "mean_abs_error": round(float(err.mean()), 2), "median_abs_error": float(err.median()),
            "within_20%_%": round(100 * float((err <= 0.2 * df["truth"]).mean()), 1),
            "median_count_over_truth": round(float((df[f"{m} | count"] / df["truth"].clip(lower=1)).median()), 2)}


def mcnemar(x, y):
    only_x, only_y = int((x & ~y).sum()), int((~x & y).sum())
    return {"only_first_right": only_x, "only_second_right": only_y,
            "p": float(binomtest(only_x, only_x + only_y, 0.5).pvalue) if only_x + only_y else 1.0}


scopes = [("all askable", q), ("E5's clean", q[q["e5_clean"] == True]),
          ("sharper photos", q[~xbd]), ("sharper photos, E5's clean", q[~xbd & (q["e5_clean"] == True)]),
          ("xBD photos", q[xbd])]
COUNT_A = pd.DataFrame([{"scope": s, **stats(df, m)} for s, df in scopes for m in METHODS])
COUNT_A.to_csv(OUT_DIR / "counting_from_table.csv", index=False)
q.drop(columns=["options"]).to_csv(OUT_DIR / "counting_per_question.csv", index=False)
clean_q = q[q["e5_clean"] == True]
PAIRED_A = {f"C4 table vs {o}": mcnemar(clean_q["C4 table | correct"].astype(bool), clean_q[f"{o} | correct"].astype(bool))
            for o in ("C4 blobs", "R11", "chain_s42")}
sharp_clean = clean_q[clean_q["source"] == "other"]
PAIRED_A["C4 table vs C4 blobs, sharper photos only"] = mcnemar(sharp_clean["C4 table | correct"].astype(bool),
                                                                 sharp_clean["C4 blobs | correct"].astype(bool))
print(COUNT_A.to_string(index=False))
print("\nExact McNemar on E5's clean questions:")
for k, v in PAIRED_A.items():
    print(f"  {k}: {v['only_first_right']} vs {v['only_second_right']}, p = {v['p']:.4g}")

# Where the rows and the blobs differ: the sharper photos, by the asked class (added after the smoke test, to explain it)
BY_CLASS = pd.DataFrame([{"class": CLASS_NAME[int(c)], **stats(g, m),
                          "count_above_truth_%": round(100 * float((g[f"{m} | count"] > g["truth"]).mean()), 1)}
                         for c, g in q[~xbd].groupby("class") for m in ("C4 table", "C4 blobs")])
BY_CLASS.to_csv(OUT_DIR / "counting_sharper_by_class.csv", index=False)
print("\nThe sharper photos, by class:")
print(BY_CLASS.to_string(index=False))

âœ“ Check 1: seed 42's table gives its saved counts on all 3,580 questions, C4's xBD rows give R11's on all 2,288 xBD questions, and both methods' answers reproduce
                     scope    method  questions  accuracy_%  largest_option_up_to_20_%  mean_abs_error  median_abs_error  within_20%_%  median_count_over_truth
               all askable  C4 table       3580       33.66                      42.10           13.78               4.0          31.6                     0.85
               all askable  C4 blobs       3580       35.34                      42.20           12.61               4.0          34.3                     0.81
               all askable       R11       3580       33.02                      41.55           13.34               4.0          32.3                     0.77
               all askable chain_s42       3580       32.88                      40.25           13.58               4.0          30.7                     0.71
                E5's clean  C4 tabl

In [3]:
# â”€â”€ Cell 3: C4's maps on the clean validation pairs, T1 v3's hot spots on them (checks 2 and 3), and the true pixels â”€â”€
val_names = sorted(p.name for p in VAL_GT.glob("*.png"))
leak40 = set(pd.read_csv(LEAKS)["validation_pair"])
assert len(val_names) == 645 and len(leak40) == 40 and leak40 <= set(val_names)
clean_val = [n for n in val_names if n not in leak40]
assert len(clean_val) == 605
calib = json.loads((STEP5 / "calibration.json").read_text(encoding="utf-8"))
assert [calib["42"]["bias"][l.capitalize()] for l in LABELS] == BIAS_42, "seed 42's validation biases differ from E7 step 2's"
inst = pd.read_csv(STEP5 / "instances_val.csv")
inst = inst[inst["found_by"] == "sam3"]
logp = inst[[f"logp_{l}_s{TABLE_SEED}" for l in LABELS]].to_numpy(np.float64)
inst = inst.assign(cls=np.where((logp + np.array(BIAS_42)).argmax(1) < 3, (logp + np.array(BIAS_42)).argmax(1) + 1, 0))
inst_of = {name: g for name, g in inst.groupby("image")}
t1_saved = pd.read_csv(T1V3 / "T1_per_image.csv").set_index("image")

names_val = clean_val
if SMOKE_TEST:
    rng = np.random.default_rng(42)
    names_val = sorted(rng.choice([n for n in clean_val if source_of(n) == "xBD"], 20, replace=False).tolist()
                       + rng.choice([n for n in clean_val if source_of(n) == "other"], 20, replace=False).tolist())
t1_check = set(names_val[::max(len(names_val) // N_T1_CHECK, 1)][:N_T1_CHECK])
cm_r11, cm_c4 = np.zeros((4, 4), np.int64), np.zeros((4, 4), np.int64)
val_rows, val_spots, t1_bad = [], [], []
for name in tqdm(names_val, desc="Validation photos"):
    gt = read_png(VAL_GT / name, flags=cv2.IMREAD_GRAYSCALE)
    r11 = read_png(VAL_R11 / name, flags=cv2.IMREAD_GRAYSCALE)
    assert r11.shape == gt.shape, name
    if source_of(name) == "xBD":
        c4 = r11
    else:
        sam = read_png(VAL_SAM / name).astype(np.int64)
        assert sam.shape == gt.shape, name
        lut = np.zeros(int(sam.max()) + 1, np.uint8)
        g = inst_of.get(name)
        if g is not None:
            lut[g["id"].to_numpy()] = g["cls"].to_numpy()
        c4 = lut[sam]
    cm_r11 += confusion(gt, r11)
    cm_c4 += confusion(gt, c4)
    spots = spot_hits(c4, gt)
    mass = damage_mass(int((c4 == 3).sum()), int((c4 == 2).sum()))
    val_rows.append({"image": name, "source": source_of(name), "gt_damaged_px": int((gt == 2).sum()),
                     "gt_destroyed_px": int((gt == 3).sum()), "mass": mass, "hot_spots": len(spots),
                     "point_on_gt_damage": sum(s["point_on_gt_damage"] for s in spots)})
    val_spots += [{"image": name, "source": source_of(name), "mass": mass, "hit": int(s["point_on_gt_damage"])} for s in spots]
    if name in t1_check:                                       # check 3: T1 v3's code on R9's map gives T1 v3's saved results
        r9 = read_png(VAL_R9 / name, flags=cv2.IMREAD_GRAYSCALE)
        mine = spot_hits(r9, gt)
        saved = t1_saved.loc[name]
        got = (len(mine), sum(s["point_on_gt_damage"] for s in mine), sum(s["point_on_gt_destroyed"] for s in mine),
               sum(s["on_gt_damage"] for s in mine), sum(s["on_gt_destroyed"] for s in mine))
        want = tuple(int(saved[c]) for c in ("hot_spots", "point_on_gt_damage", "point_on_gt_destroyed", "on_gt_damage", "on_gt_destroyed"))
        if got != want:
            t1_bad.append((name, got, want))
VAL_C4 = pd.DataFrame(val_rows)
VAL_SPOTS = pd.DataFrame(val_spots)
VAL_C4.to_csv(OUT_DIR / "val_c4_hotspots_per_image.csv", index=False)
assert not t1_bad, f"check 3 failed on {len(t1_bad)} photos, e.g. {t1_bad[:2]}"
print(f"âœ“ Check 3: T1 v3's code on R9's validation maps gives T1 v3's saved results on all {len(t1_check)} checked photos")

cand = pd.read_csv(S2C / "candidates_validation.csv")
c4_row = cand[cand["candidate"].str.startswith("C4")].iloc[0]
CHECK2 = pd.DataFrame([
    {"what": "R11 with TTA: destroyed IoU", "rebuilt": ious(cm_r11)["destroyed_iou"], "published": 0.2692},
    {"what": "R11 with TTA: mIoU", "rebuilt": ious(cm_r11)["miou"], "published": 0.5805},
    {"what": "C4, seed 42: destroyed IoU", "rebuilt": ious(cm_c4)["destroyed_iou"], "published": float(c4_row["destroyed_iou_s42"])}])
CHECK2["difference"] = (CHECK2["rebuilt"] - CHECK2["published"]).round(4)
print(CHECK2.to_string(index=False))
if SMOKE_TEST:
    print("Smoke test: check 2 needs all 605 clean validation pairs, so it isn't applied")
else:
    assert (CHECK2["difference"].abs() <= TOL).all(), "check 2 failed: the rebuilt validation maps don't give E7 step 2c's scores"
    print("âœ“ Check 2: C4's rebuilt validation maps give E7 step 2c's scores within 0.0005")

# The test photos' true pixels (E6's masks), for check B
pairs = pd.read_csv(E6 / "test_pairs.csv")
test_hot = pd.read_csv(S3 / "hotspots_per_image.csv")
if SMOKE_TEST:
    test_hot = test_hot.sample(100, random_state=42)
mask_file = dict(zip(pairs["after_image"], pairs["mask_file"]))
MASKS = zipfile.ZipFile(E6 / "test_masks_4class.zip")
test_hot["gt_damaged_px"] = [int((read_png(MASKS, mask_file[n], cv2.IMREAD_GRAYSCALE) == 2).sum())
                             for n in tqdm(test_hot["image"], desc="Test masks")]
records = json.loads((S3 / "hotspots.json").read_text(encoding="utf-8"))
REC = {r["image"]: r for r in records}
test_hot["mass"] = [damage_mass(REC[n]["pred_destroyed_px"], REC[n]["pred_damaged_px"]) for n in test_hot["image"]]
print(f"âœ“ {len(VAL_C4)} validation photos with C4's hot spots; {len(test_hot)} test photos with their true damaged pixels")

Validation photos:   0%|          | 0/605 [00:00<?, ?it/s]

âœ“ Check 3: T1 v3's code on R9's validation maps gives T1 v3's saved results on all 60 checked photos
                       what  rebuilt  published  difference
R11 with TTA: destroyed IoU   0.2692     0.2692         0.0
         R11 with TTA: mIoU   0.5805     0.5805         0.0
 C4, seed 42: destroyed IoU   0.3574     0.3574         0.0
âœ“ Check 2: C4's rebuilt validation maps give E7 step 2c's scores within 0.0005


Test masks:   0%|          | 0/2007 [00:00<?, ?it/s]

âœ“ 605 validation photos with C4's hot spots; 2007 test photos with their true damaged pixels


In [4]:
# â”€â”€ Cell 4: Check B, why the hot spots fall short: the maps or the photos â”€â”€
r9 = t1_saved.loc[[n for n in VAL_C4["image"]]].reset_index()
r9["gt_damaged_px"] = VAL_C4["gt_damaged_px"].to_numpy()
assert (r9["gt_destroyed_px"].to_numpy() == VAL_C4["gt_destroyed_px"].to_numpy()).all()


def group(df):
    return np.where(df["gt_destroyed_px"] > 0, "with destroyed",
                    np.where(df["gt_damaged_px"] > 0, "damaged only", "no true damage"))


SETS = [("R9's maps, validation", r9), ("C4's maps, validation", VAL_C4), ("C4's maps, test", test_hot)]
rows = []
for set_name, df in SETS:
    df = df.assign(group=group(df))
    for src, d in [("all", df), *df.groupby("source")]:
        no_dest = d[d["group"] != "with destroyed"]
        for grp, g in [*d.groupby("group"), ("without destroyed", no_dest)]:
            n_spots = int(g["hot_spots"].sum())
            rows.append({"maps_and_photos": set_name, "source": src, "group": grp, "photos": len(g),
                         "share_of_photos_without_destroyed": round(len(g) / max(len(no_dest), 1), 3) if grp != "with destroyed" else None,
                         "hot_spots": n_spots, "hot_spots_per_photo": round(n_spots / max(len(g), 1), 2),
                         "hit_share": round(int(g["point_on_gt_damage"].sum()) / n_spots, 3) if n_spots else None})
GROUPS = pd.DataFrame(rows)
GROUPS.to_csv(OUT_DIR / "hotspot_groups.csv", index=False)
print(GROUPS.to_string(index=False))


def share(set_name, src="all", grp="without destroyed"):
    r = GROUPS[(GROUPS["maps_and_photos"] == set_name) & (GROUPS["source"] == src) & (GROUPS["group"] == grp)]
    return float(r["hit_share"].iloc[0]) if len(r) and r["hit_share"].iloc[0] is not None else float("nan")


causes = []
for src in ("all", "xBD", "other"):
    a, b, c = (share(s, src) for s, _ in SETS)
    maps, photos = b - a, c - b
    reading = ("both" if abs(maps) > 0.05 and abs(photos) > 0.05 else "the maps" if abs(maps) >= abs(photos) else "the photos")
    causes.append({"source": src, "R9_val": a, "C4_val": b, "C4_test": c, "maps_change": round(maps, 3),
                   "photos_change": round(photos, 3), "main_cause_by_the_rule": reading})
CAUSES = pd.DataFrame(causes)
CAUSES.to_csv(OUT_DIR / "hotspot_causes.csv", index=False)
print("\nOn photos without destroyed buildings, the hit share at each stage:")
print(CAUSES.to_string(index=False))

      maps_and_photos source             group  photos  share_of_photos_without_destroyed  hot_spots  hot_spots_per_photo  hit_share
R9's maps, validation    all      damaged only     145                              0.344        221                 1.52      0.783
R9's maps, validation    all    no true damage     277                              0.656         75                 0.27      0.000
R9's maps, validation    all    with destroyed     183                                NaN        332                 1.81      0.852
R9's maps, validation    all without destroyed     422                              1.000        296                 0.70      0.584
R9's maps, validation  other      damaged only      16                              0.254          7                 0.44      0.286
R9's maps, validation  other    no true damage      47                              0.746         10                 0.21      0.000
R9's maps, validation  other    with destroyed      22               

In [5]:
# â”€â”€ Cell 5: Check C, the stricter rule (T1 v4): chosen on validation, confirmed once on the test pairs â”€â”€
GATE_VAL = gate_table(VAL_SPOTS, GATE_GRID)
GATE = choose_gate(GATE_VAL, KEEP_HITS_VAL)
GATE_VAL.to_csv(OUT_DIR / "gate_validation.csv", index=False)
print("Validation (C4's maps, clean pairs):")
print(GATE_VAL.to_string(index=False))
print(f"\nâœ“ The rule's choice: G = {GATE} px (the largest that keeps at least {KEEP_HITS_VAL:.0%} of the validation hits)")

test_spots = pd.DataFrame([{"image": n, "source": REC[n]["source"], "mass": m, "hit": int(s["point_on_gt_damage"]),
                            "with_destroyed": bool(d > 0)}
                           for n, m, d in zip(test_hot["image"], test_hot["mass"], test_hot["gt_destroyed_px"])
                           for s in REC[n]["hot_spots"]])
rows = []
for scope, d in [("all", test_spots), ("with destroyed", test_spots[test_spots["with_destroyed"]]),
                 ("without destroyed", test_spots[~test_spots["with_destroyed"]]), *test_spots.groupby("source")]:
    for version, g in (("v3", 0), ("v4", GATE)):
        kept = d[d["mass"] >= g]
        rows.append({"scope": scope, "version": version, "gate_px": g, "hot_spots": len(kept), "hits": int(kept["hit"].sum()),
                     "hit_share": round(float(kept["hit"].mean()), 3) if len(kept) else None,
                     "photos_with_hot_spots": kept["image"].nunique()})
GATE_TEST = pd.DataFrame(rows)
v3 = GATE_TEST[(GATE_TEST["scope"] == "all") & (GATE_TEST["version"] == "v3")].iloc[0]
v4 = GATE_TEST[(GATE_TEST["scope"] == "all") & (GATE_TEST["version"] == "v4")].iloc[0]
HITS_KEPT = v4["hits"] / max(v3["hits"], 1)
GAIN = (v4["hit_share"] or 0) - v3["hit_share"]
PASSES = bool(GATE > 0 and HITS_KEPT >= KEEP_HITS_TEST and GAIN >= MIN_GAIN)
GATE_TEST["hits_kept"] = (GATE_TEST["hits"] / GATE_TEST.groupby("scope")["hits"].transform("max").clip(lower=1)).round(3)
GATE_TEST.to_csv(OUT_DIR / "gate_test.csv", index=False)
print("\nTest (C4's maps, clean pairs), once:")
print(GATE_TEST.to_string(index=False))
if not SMOKE_TEST:
    assert abs(v3["hit_share"] - V3_TEST_SHARE) <= 0.001, "v3's test hit share doesn't match E7 step 3's"
print(f"\nv4 keeps {HITS_KEPT:.1%} of the test hits and moves the hit share by {GAIN:+.3f}: "
      f"{'PASSES' if PASSES else 'does not pass'} (needs at least {KEEP_HITS_TEST:.0%} and +{MIN_GAIN})")

if PASSES:
    v4_records = []
    for r in records:
        r = dict(r)
        r["gate_px"], r["damage_mass_px"] = GATE, damage_mass(r["pred_destroyed_px"], r["pred_damaged_px"])
        if r["damage_mass_px"] < GATE:
            r["hot_spots"], r["safe_areas"] = [], []          # as T1 v3 does for a photo with no damage
        v4_records.append(r)
    with open(OUT_DIR / "hotspots_v4.json", "w", encoding="utf-8") as f:
        json.dump(v4_records, f, indent=1)
    print(f"âœ“ Wrote hotspots_v4.json: {sum(1 for r in v4_records if r['damage_mass_px'] < GATE)} of {len(v4_records)} photos lose their hot spots")

Validation (C4's maps, clean pairs):
 gate_px  hot_spots  hits  hit_share  photos_with_hot_spots  hits_kept
       0        726   511      0.704                    326      1.000
     100        726   511      0.704                    326      1.000
     200        724   511      0.706                    324      1.000
     400        703   501      0.713                    303      0.980
     800        663   481      0.725                    265      0.941
    1600        591   443      0.750                    223      0.867
    3200        500   381      0.762                    182      0.746
    6400        375   289      0.771                    133      0.566
   12800        257   196      0.763                     88      0.384

âœ“ The rule's choice: G = 400 px (the largest that keeps at least 95% of the validation hits)

Test (C4's maps, clean pairs), once:
            scope version  gate_px  hot_spots  hits  hit_share  photos_with_hot_spots  hits_kept
              all     

In [6]:
# â”€â”€ Cell 6: Against the expectation; README and config â”€â”€
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


def a_row(scope, m):
    return COUNT_A[(COUNT_A["scope"] == scope) & (COUNT_A["method"] == m)].iloc[0]


gain_clean = a_row("E5's clean", "C4 table")["accuracy_%"] - a_row("E5's clean", "C4 blobs")["accuracy_%"]
gain_sharp = a_row("sharper photos", "C4 table")["accuracy_%"] - a_row("sharper photos", "C4 blobs")["accuracy_%"]
ratio_sharp = a_row("sharper photos", "C4 table")["median_count_over_truth"]
cause_all = CAUSES[CAUSES["source"] == "all"].iloc[0]
EXPECT = {
    "all four checks pass": True,
    "A: median count / truth on the sharper photos 0.8-0.9": bool(0.8 <= ratio_sharp <= 0.9),
    "A: accuracy on the sharper photos up 1-4 points": bool(1 <= gain_sharp <= 4),
    "A: C4's accuracy on E5's clean questions up 0.5-1.5 points": bool(0.5 <= gain_clean <= 1.5),
    "B: mostly the maps (C4 below 0.45 on the same validation photos)": bool(cause_all["main_cause_by_the_rule"] in ("the maps", "both")
                                                                             and cause_all["C4_val"] < 0.45),
    "C: G between 400 and 3,200 px": bool(400 <= GATE <= 3200),
    "C: test hit share 0.60-0.68 with at least 90% of hits kept": bool(0.60 <= (v4["hit_share"] or 0) <= 0.68 and HITS_KEPT >= 0.9),
}
RUN_MIN = round((time.time() - t_start) / 60, 1)
config = {"task": "E7 step 3b", "smoke_test": SMOKE_TEST, "questions": len(q), "validation_photos": len(VAL_C4),
          "test_photos": len(test_hot), "table_seed": TABLE_SEED, "gate_grid_px": GATE_GRID, "gate_px": GATE,
          "v4_hits_kept": round(float(HITS_KEPT), 3), "v4_gain": round(float(GAIN), 3), "v4_passes": PASSES,
          "paired_tests": PAIRED_A, "expectation": EXPECT, "run_time_min": RUN_MIN,
          "versions": {"opencv": cv2.__version__, "numpy": np.__version__, "pandas": pd.__version__, "python": sys.version.split()[0]}}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, default=str), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    f"# E7 step 3b: three follow-up checks on C4's outputs{' (SMOKE TEST)' if SMOKE_TEST else ''}", "",
    "The questions, the methods, the checks and the expectation, fixed before the run, are in the notebook's first cell "
    "([`e7-step3b-three-checks.ipynb`](../../e7-step3b-three-checks.ipynb)). Run on a PC, no GPU, in "
    f"{RUN_MIN} min.", "",
    "## Check A: counting from the table's rows", "",
    "Each method's count is the number of rows (or blobs) of the asked class on the photo; the answer is the nearest "
    "option. `C4 table` counts the rows of `buildings_c4.sqlite` (one per SAM 3 outline on the sharper photos, one per "
    "patch of R11's pixels on xBD photos); `C4 blobs` is E7 step 3's figure from E7 step 2's joined blobs.", "",
    as_markdown(COUNT_A), "",
    "Exact McNemar on E5's clean questions: " + "; ".join(f"{k}: {v['only_first_right']} vs {v['only_second_right']}, "
                                                           f"p = {v['p']:.4g}" for k, v in PAIRED_A.items()), "",
    "The sharper photos, by the asked class (added after the smoke test showed the drop, to explain it):", "",
    as_markdown(BY_CLASS), "",
    "## Check B: why the hot spots fall short on photos without destroyed buildings", "",
    "A hit is a hot spot whose marked point is within 20 px of a truly damaged or destroyed building. Groups: photos "
    "with truly destroyed buildings; with only damaged ones; with no true damage.", "",
    as_markdown(GROUPS), "",
    "On photos without destroyed buildings, from R9's maps to C4's on the same validation photos (the maps change), "
    "then to C4's on the test photos (the photos change):", "",
    as_markdown(CAUSES), "",
    "## Check C: a stricter rule for when a photo gets hot spots (T1 v4)", "",
    f"A photo gets hot spots only if its map marks at least G px of damage (destroyed 1, damaged 0.5). Chosen on "
    f"validation: **G = {GATE} px**.", "",
    as_markdown(GATE_VAL), "",
    "Confirmed once on the clean test pairs:", "",
    as_markdown(GATE_TEST), "",
    f"**v4 {'passes' if PASSES else 'does not pass'}:** it keeps {HITS_KEPT:.1%} of the test hits (needs at least "
    f"{KEEP_HITS_TEST:.0%}) and moves the hit share by {GAIN:+.3f} (needs at least +{MIN_GAIN}). "
    + ("`hotspots_v4.json` holds the gated hot spots; switching T2 step 5 to it is Abrar's call." if PASSES
       else "T1 v3 stays."), "",
    "## Checks", "",
    "1. Seed 42's table gives its saved counts; C4's xBD rows give R11's; the answers reproduce: passed.",
    "2. C4's rebuilt validation maps give E7 step 2c's scores within 0.0005: " + ("not applied (smoke test)." if SMOKE_TEST else "passed."), "",
    as_markdown(CHECK2), "",
    f"3. T1 v3's code on R9's validation maps gives T1 v3's saved results on {len(t1_check)} photos: passed.",
    "4. The gate's self-checks: passed.", "",
    "## Against the expectation, fixed before the run", "",
    *[f"- {k}: **{'yes' if v else 'no'}**" for k, v in EXPECT.items()], "",
    "## Files", "",
    "`counting_from_table.csv`, `counting_sharper_by_class.csv`, `counting_per_question.csv`, `hotspot_groups.csv`, `hotspot_causes.csv`, "
    "`val_c4_hotspots_per_image.csv`, `gate_validation.csv`, `gate_test.csv`, `config.json`"
    + (", `hotspots_v4.json`" if PASSES else "")]), encoding="utf-8")
print("\n".join(f"- {k}: {'yes' if v else 'no'}" for k, v in EXPECT.items()))
print(f"\nâœ“ Wrote the README, the CSVs and config.json to {OUT_DIR} in {RUN_MIN} min")

- all four checks pass: yes
- A: median count / truth on the sharper photos 0.8-0.9: yes
- A: accuracy on the sharper photos up 1-4 points: no
- A: C4's accuracy on E5's clean questions up 0.5-1.5 points: no
- B: mostly the maps (C4 below 0.45 on the same validation photos): no
- C: G between 400 and 3,200 px: yes
- C: test hit share 0.60-0.68 with at least 90% of hits kept: no

âœ“ Wrote the README, the CSVs and config.json to D:\CSE499AB_project\Essential_task_delegation\Champion_Task\E7-results\step3b in 6.0 min
